# Post-ICA autoreject and interpolation

Separated from the ICA apply step: autoreject takes ~20 min while applying
a decomposition takes seconds, so keeping them together made re-running a
manual ICA decision needlessly expensive.


In [ ]:
subject_id = "170"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. derivatives_root -> a test folder
path_overrides = None


In [ ]:
import mne
from mne.preprocessing import ICA
import autoreject
from pathlib import Path
import pandas as pd
from datetime import datetime
import json
import os

from spectral.viz import plot_step, plot_bad_channels, plot_epochs
from spectral.specparam import specparam2pandas
from spectral.utils import ProjectPaths,print_timestamp
from spectral.ica import compute_ica_limited, plot_annotated_topographies, plot_component_properties,get_manual_ica_list
from mne_icalabel.gui import label_ica_components
from mne_icalabel import label_components


""" config = read_parameters()
my_paths = config_project(subject)
project_path = my_paths["project_path"]
figures_path = my_paths["figures_path"]
specparam_path = my_paths["specparam_path"]

Path(specparam_path).mkdir(parents=True, exist_ok=True)
 """


paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
# Create all directories
paths.create_directories()

#paths = ProjectPaths(subject)
# Create all directories
#paths.create_directories()

# This is analysis output, so it goes in the analysis folder
specparam_path = paths.specparam
project_path = paths.root
figures_path = paths.figures
preprocessed_path = paths.preprocessed
epochs_path = paths.epochs
# Print paths to verify
print_timestamp("Setting up project paths")
paths.show()

mne.viz.set_browser_backend("matplotlib")
# mne.viz.set_browser_backend("qt")
mne.set_config("MNE_BROWSER_THEME", "light")
# matplotlib.use("Agg")
#print_date_time()

In [ ]:
clean_raw = mne.io.read_raw_fif(
    f"{paths.preprocessed}/sub-{subject_id}_annotated_filtered_raw.fif", preload=True
)

total_duration = clean_raw.times[-1]
clean_raw = clean_raw.crop(tmin=3.0, tmax=total_duration - 3, include_tmax=True)

report = mne.Report(title=f"Report for subject {subject_id}", subject=subject_id)
report.add_raw(raw=clean_raw, title="Raw", psd=True, butterfly=True, scalings='auto')

# --- HANDLES CASE WITH NO BAD CHANNELS ---
if clean_raw.info["bads"]:
    print(f"Plotting {len(clean_raw.info['bads'])} bad channels for the report.")
    
    # We use a try-except here because plotting bads can occasionally 
    # throw a ValueError if channels were marked bad but then dropped.
    try:
        bad_channel_plot = clean_raw.plot(
            picks=clean_raw.info["bads"],
            duration=min(300.0, total_duration - 6), # Ensure duration isn't longer than file
            scalings=dict(eeg=1e-4),
            show_scrollbars=False,
            show=False # Prevent popup in headless environments
        )
        report.add_figure(bad_channel_plot, title="Rejected bad channels")
    except Exception as e:
        print(f"Could not plot bad channels: {e}")
else:
    print("No bad channels detected. Skipping bad channel plot.")
    # Optional: Add a text note to the report instead of a plot
    report.add_html(
        html="<p><b>Note:</b> No bad channels were detected or marked for this subject.</p>",
        title="Rejected bad channels"
    )

In [ ]:
epochs_good = mne.read_epochs(
    f"{paths.epochs}/sub-{subject_id}_good_epochs-epo.fif"
)

In [ ]:
from spectral.ica import read_ica_application, describe_ica_application

# Verify the record against the saved cleaned epochs, including any FIF splits.
# A missing/old record requires rerunning 04; do not infer it from today's review.
epochs_source = paths.analysis / f"sub-{subject_id}_clean_ica-epo.fif"
application = read_ica_application(
    paths.analysis / f"sub-{subject_id}_ica-application.json",
    epochs_path=epochs_source)
epochs_clean = mne.read_epochs(epochs_source, preload=True)
print(describe_ica_application(application))


In [ ]:
# Post-ICA autoreject. Parameters come from settings.toml ([interpolate]);
# they used to be hardcoded here and are unchanged.
_ip = paths.config.get("interpolate", {})

auto_reject_pre_ica = autoreject.AutoReject(
    n_interpolate=_ip.get("autoreject_n_interpolate", [1, 2, 4, 8, 32, 64]),
    n_jobs=-1,
    random_state=_ip.get("autoreject_random_state", 100),
    thresh_method=_ip.get("autoreject_thresh_method", "bayesian_optimization"),
    verbose=False,
).fit(epochs_clean)
print("fitting finished")
epochs_ar, reject_log = auto_reject_pre_ica.transform(
   epochs_clean, return_log=True
)

# Guard: this pass dropped ALL 173 epochs on sub-114, and the interpolate call
# below then raised on an empty object, failing the subject outright. Keep the
# uncleaned epochs rather than losing the recording, and record that we did.
_min_epochs = _ip.get("min_epochs_after_ar", 10)
ar_applied = len(epochs_ar) >= _min_epochs
if not ar_applied:
    print(f"WARNING autoreject kept {len(epochs_ar)} of {len(epochs_clean)} epochs "
          f"(< {_min_epochs}). Keeping the un-rejected epochs instead.")
    report.add_html(
        html=f"<p><b>Autoreject skipped.</b> The post-ICA pass kept only "
             f"{len(epochs_ar)} of {len(epochs_clean)} epochs, below the "
             f"min_epochs_after_ar = {_min_epochs} floor, so its rejection was "
             f"discarded and the un-rejected epochs carried forward. "
             f"This subject needs review.</p>",
        title="Autoreject skipped",
    )
    epochs_ar = epochs_clean.copy()
else:
    print(f"autoreject kept {len(epochs_ar)} of {len(epochs_clean)} epochs")

# depending of the resampling - different number of epochs are rejected

reject_plot = reject_log.plot("vertical")
report.add_figure(
    reject_plot, title="Interpolated epochs - AutoReject overview")

# Add the last part to the report
report.add_figure(
    reject_plot, title="Autoreject - Rejected and interpolated epochs after ICA"
)


In [ ]:
epochs_interpolated = epochs_ar.copy().interpolate_bads(exclude=["VREF"])
epochs_interpolated.save(
    f"{paths.analysis}/sub-{subject_id}_interpolated-epo.fif", overwrite=True
)

In [ ]:
# epochs_interpolated
epochs_plot, epochs_timeseries, epochs_plot_psd = plot_epochs(
    epochs_interpolated, figures_path, subject_id, stage="final_interpolared_epochs"
)
# Add figures to the report
report.add_figure(epochs_plot, title="Interpolated epochs")
report.add_figure(epochs_plot_psd, title="Interpolated epochs - PSD")

In [ ]:
# Counts and provenance come from the recorded ICA application.
# Later review edits cannot relabel already-cleaned data.
_excluded = application["excluded_components"]

# Create metadata dataframe
now = datetime.now()
metadata_df = pd.DataFrame({
    "subject_id": [subject_id],
    "timestamp": [now],
    "nr_interpolated_channels": [len(epochs_ar.info["bads"])],
    "nr_dropped_ica": [len(_excluded)],
    "nr_retained_ica": [application["n_components"] - len(_excluded)],
    "total_ica_components": [application["n_components"]],
    "nr_epochs_after_ar": [len(epochs_ar)],
    "autoreject_applied": [ar_applied],   # False = kept un-rejected epochs, needs review
    "bad_channels": [",".join(epochs_ar.info["bads"])],
    "exclusion_source": [application["exclusion_source"]],
    "ica_fingerprint": [application["ica_fingerprint"]],
})

# Save metadata to CSV
metadata_path = f"{paths.specparam}/sub-{subject_id}_ica_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print(f"Metadata saved to: {metadata_path}")


In [ ]:
report.save(
    f"{paths.reports}/sub-{subject_id}_report_05_interpolate.html", overwrite=True
)
